In [175]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split

In [176]:
# transformers and pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# ml models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [177]:
df = pd.read_csv('data/breast_cancer_diagnostic/wdbc.data', header=None)

In [178]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [179]:
X = df.iloc[:, 2:].values
y = df.iloc[:, 1].values

In [180]:
y = le.fit_transform(y)
le.classes_

array(['B', 'M'], dtype=object)

In [181]:
le.transform(['M', 'B'])

array([1, 0])

In [182]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [183]:
pipe_lr = make_pipeline(StandardScaler(),
                        PCA(n_components=2),
                        #SVC(random_state=1, kernel='linear', C=1.0),
                        LogisticRegression(random_state=1, C=1.0)
)

In [184]:
pipe_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",2
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'au

In [185]:
y_pred = pipe_lr.predict(X_test)

In [186]:
test_accuracy = pipe_lr.score(X_test, y_test)
print(f'Test Accuracy: {test_accuracy:.3f}')

Test Accuracy: 0.947


In [187]:
kFold = StratifiedKFold(n_splits=10).split(X_train, y_train)

In [188]:
scores = []
for k, (train, test) in enumerate(kFold):
    pipe_lr.fit(X_train[train], y_train[train])
    score = pipe_lr.score(X_train[test], y_train[test])
    scores.append(score)
    print(f'Fold: {k+1}, Class dist.: {np.bincount(y_train[train])}, '
          f'Accuracy: {score:.3f}')

Fold: 1, Class dist.: [256 153], Accuracy: 0.891
Fold: 2, Class dist.: [256 153], Accuracy: 0.978
Fold: 3, Class dist.: [256 153], Accuracy: 0.978
Fold: 4, Class dist.: [256 153], Accuracy: 0.913
Fold: 5, Class dist.: [256 153], Accuracy: 0.935
Fold: 6, Class dist.: [257 153], Accuracy: 0.978
Fold: 7, Class dist.: [257 153], Accuracy: 0.933
Fold: 8, Class dist.: [257 153], Accuracy: 0.956
Fold: 9, Class dist.: [257 153], Accuracy: 0.978
Fold: 10, Class dist.: [257 153], Accuracy: 0.956


In [189]:
mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\nMean Accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')


Mean Accuracy: 0.950 +/- 0.029


In [190]:
scores = cross_val_score(estimator=pipe_lr,
                         X=X_train, y=y_train, cv=10, n_jobs=1)
print(f'CV accuracy scores: {scores}')

CV accuracy scores: [0.89130435 0.97826087 0.97826087 0.91304348 0.93478261 0.97777778
 0.93333333 0.95555556 0.97777778 0.95555556]
